# ResNet-18 — geheugen en snelheid op de laptop

**Korte synthetische controle · globale batch 256 · vier groepen van 64 · vijf W/A-precisies**

Deze notebook toont de echte controle-uitvoer. De nieuwe proef gebruikt de afgesproken ResNet-18-reconstructie met de oorspronkelijke Any-Precision-quantizers, FP32 en SGD. Het doel is te beoordelen of een volgende ImageNet-pilot praktisch haalbaar lijkt.

Er zijn **geen ImageNet-epochs of datasetaccuracies** gemeten. De eerdere ResNet-50-controles dienen als hardwarecontext. Model en learning rate verschillen tussen de nieuwe en eerdere proeven; de vergelijking is dus geen proef waarin maar één instelling verandert.


In [ ]:
from pathlib import Path
import importlib.util
from IPython.display import display

RUN_DIRECTORY = ""
run_directory = Path(RUN_DIRECTORY).expanduser().resolve() if RUN_DIRECTORY else Path.cwd()
helper = run_directory / "notebook_support.py"
if not helper.is_file():
    raise FileNotFoundError("Kies de resultatenmap met run_report.json, baseline_results.json en notebook_support.py.")
spec = importlib.util.spec_from_file_location("resnet18_check_notebook_support", helper)
ns = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ns)

run = ns.load_run(run_directory)
print("Resultatenmap:", run_directory)
print("Totale rapportstatus:", run["report"].get("status", "ONBEKEND"))
print(ns.outcome_text(run))


## 1. Opzet en controle van de implementatie

De architectuurbasis is de **ImageNet-ResNet-18 uit `trunc_models` van TruncQuant**, met vier stages en `[2, 2, 2, 2]` BasicBlocks. De hoofdbranches gebruiken BN-banken per precisie; stem- en projectie-BN zijn gedeeld. Die architectuurkeuze is controleerbaar, maar is geen bewijs van identiteit met de oorspronkelijke ResNet-18-implementatie uit de Any-Precision-paper. Die implementatie is niet aangetroffen in de geraadpleegde bronnen.

We gebruiken gewone afrondende quantisatie voor W/A **1, 2, 4, 8 en 32 bit**, met FP32-rekentensors. Er is geen MX, truncatie, AMP of activation checkpointing. De afgesproken globale batch is **256**, de vier lokale groepen bevatten elk **64** beelden. De LR is **0,3**, momentum **0,9** en weight decay **0,00001**. De precieze oorspronkelijke ResNet-18-batch en weight decay zijn onbevestigd; hier zijn het vastgelegde startkeuzes.

De CPU-zelftests controleren implementatie en trainingssemantiek. Een CPU-PASS bewijst geen GPU-geheugenverbruik, GPU-snelheid of bitexact gelijke CUDA-uitkomst.


In [ ]:
display(ns.correctness_table(run))
config = run["report"].get("config", {})
if config:
    display(ns.pd.DataFrame([{"Instelling": key, "Waarde": str(value)} for key, value in config.items()]))


## 2. De afzonderlijke meetgegevens

Een gemeten stap is **één globale update**, inclusief alle vijf precisies, twintig forward/backward-paren en één SGD-update. Tijden zijn seconden; geheugen wordt uitgedrukt in **GiB = 2³⁰ bytes**.

De twee oorspronkelijke ResNet-50-controles zijn voltooid. De latere `benchmark=False`-proef is een echte **TIMEOUT** met één voltooide gemeten stap; die blijft zichtbaar als gedeeltelijk resultaat. Ontbrekende metingen worden niet als nul weergegeven. Een CPU-zelftest zonder GPU-proef krijgt **GEEN_GPU_METING**.


In [ ]:
comparison = ns.comparison_table(run)
display(comparison.round(3))
display(ns.comparison_context_table(run))
if run["report"].get("comparison_context"):
    print("Vergelijkingscontext uit het rapport:", run["report"]["comparison_context"])


## 3. Staptijden na de opwarming

Iedere volledige proef bevat één opwarmupdate en twee gemeten updates. De twee metingen blijven afzonderlijk zichtbaar. Een open punt markeert een voltooide stap uit een onvoltooide proef; de afgebroken volgende stap krijgt geen geschatte duur.

De opwarming bevat extra controles en synchronisatie. Tijdens de eerste optimizer-update wordt ook de SGD-momentumstaat aangemaakt. Daarom behandelen we de opwarmduur niet als een derde vergelijkbare meting.


In [ ]:
fig_times = ns.plot_step_times(run)
display(fig_times)
ns.plt.close(fig_times)


## 4. Vastgelegde geheugenpieken

**Allocated** is geheugen voor PyTorch-tensors; **reserved** is geheugen dat de PyTorch-allocator beheert. Dit zijn **geen directe metingen van fysieke VRAM-residentie of Windows-gedeeld geheugen**. De capaciteitslijn geeft context en bewijst op zichzelf geen paging of oorzaak van vertraging.

Bij een onvoltooide proef is alleen de laatst vastgelegde piek bekend. Een gearceerde staaf toont die deelmeting; de piek van een afgebroken update kan onbekend zijn. De tabel eronder toont geheugenstanden per update, voor zover gemeten.


In [ ]:
fig_memory = ns.plot_memory(run)
display(fig_memory)
ns.plt.close(fig_memory)
updates = ns.update_memory_table(run)
if updates.empty:
    print("Geen geheugenstanden per update in dit rapport.")
else:
    display(updates.round(3))


## 5. Voorzichtige projectie van de rekentijd

Alleen na een volledige ResNet-18-PASS worden onderstaande scenario's berekend. Bij 1.281.167 ImageNet-trainingsbeelden en batch 256 zijn er **5.005 updates per epoch**. We vermenigvuldigen dat aantal met de snelste, middelste en langzaamste van de twee gemeten stapduren.

**Dit is een rekenkundig scenario, geen gemeten epochduur en geen betrouwbaarheidsinterval.** Dataladen, JPEG-decodering, augmentatie, datasetvalidatie en checkpointopslag vallen erbuiten. Ook langdurige thermische effecten, geheugendruk en stabiliteit zijn met twee stappen niet vastgesteld. De volledige 80-epochtraining wordt hiermee niet automatisch haalbaar verklaard of gestart.


In [ ]:
projection = ns.projection_table(run)
if projection.empty:
    print("Geen projectie: er zijn geen twee voltooide ResNet-18-metingen met GPU-status PASS.")
else:
    display(projection.round(3))
    print("Alleen de gemeten GPU-updates geëxtrapoleerd; geen volledige trainingstijdmeting.")


## 6. Betekenis voor de volgende stap

- **PASS met voldoende geheugenruimte en bruikbare staptijden:** aanleiding om een echte pilot te overwegen, inclusief dataladen en evaluatie.
- **PASS met nauwelijks geheugenruimte of zeer lange / sterk uiteenlopende staptijden:** technisch voltooid, maar de praktische haalbaarheid blijft twijfelachtig.
- **TIMEOUT:** de begrensde proef is gestopt; dit bewijst geen geheugen-OOM.
- **OOM of FAIL:** eerst de fout en gedeeltelijke gegevens beoordelen.

De paper noemt voor een volledige any-precisiontraining 80 epochs met SGD, LR 0,3 en verlagingen na 45, 60 en 70 epochs. De architectuur, batch- en BN-keuzes van onze reconstructie blijven expliciet in het protocol staan. Een eventuele volledige training begint later opnieuw vanaf epoch 0; deze diagnose levert geen trainingscheckpoint.


In [ ]:
test = run["report"].get("gpu_check") or {}
for field in ("error", "last_progress", "elapsed_seconds", "timeout_seconds", "optimizer_updates_completed"):
    value = test.get(field)
    if value is not None:
        print(f"{field}: {value}")
if run["report"].get("error"):
    print("Rapportfout:", run["report"]["error"])
print(ns.outcome_text(run))


## Bronnen en herkomst

- Meetgegevens: `run_report.json` (nieuwe proef in `gpu_check`) en `baseline_results.json` (drie echte eerdere gebruikerscontroles).
- Implementatiekeuzes en interpretatie: `PROTOCOL.md` in het pakket; bronhashes in het runrapport.
- [Any-Precision-paper](https://arxiv.org/html/1911.07346v2).
- [Oorspronkelijke Any-Precision-code](https://github.com/SHI-Labs/Any-Precision-DNNs).
- [TruncQuant-code](https://github.com/a2jinhee/TruncQuant).
- [PyTorch 2.10: DataParallel](https://docs.pytorch.org/docs/2.10/generated/torch.nn.DataParallel.html).
- [PyTorch 2.10: GPU-geheugenbeheer](https://docs.pytorch.org/docs/2.10/notes/cuda.html#memory-management).

De notebook analyseert bestaande rapporten. Zij voert geen GPU-proef of training uit.
